# 01 · COVIDx CXR-3A data exploration

Explore official metadata and images that are actually accessible in this runtime. Upstream documented counts and local validation counts are shown separately. Run notebook 02 first to create `patients.csv`, or point the variables below to an equivalent verified metadata file. No model is trained here.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import Image as DisplayImage, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

from covid_xray.config import CLASSES  # noqa: E402
from covid_xray.data import audit_metadata_file  # noqa: E402
from covid_xray.eda import create_dataset_report  # noqa: E402

# Change these environment variables for Google Drive or another local mount.
DATA_DIR = Path(os.environ.get('COVIDX_DATA_DIR', ROOT / 'data/raw/covidx')).expanduser()
METADATA = Path(os.environ.get('COVIDX_METADATA', ROOT / 'data/processed/patients.csv')).expanduser()
VALIDATED = ROOT / 'data/processed/validated_metadata.csv'
OUTPUT_DIR = ROOT / 'artifacts/results/eda'
QUALITY_REPORT = OUTPUT_DIR / 'data_quality_report.json'
if not METADATA.is_file():
    raise FileNotFoundError('Run 02_preprocessing.ipynb first or set COVIDX_METADATA.')

## 2. Metadata loading and documented record counts

In [ ]:
# These rows come from the pinned official label files, not from file discovery.
metadata = pd.read_csv(METADATA, dtype=str, keep_default_na=False)
display(metadata.head())
metadata_counts = pd.crosstab(metadata['label'], metadata['original_partition']).reindex(CLASSES, fill_value=0)
display(metadata_counts)
print('Upstream metadata records loaded:', len(metadata))

## 3. Validate accessible images

In [ ]:
# Missing, unsupported and corrupt files remain visible in the audited table.
audited_data, quality_report = audit_metadata_file(
    METADATA, DATA_DIR, VALIDATED, QUALITY_REPORT
)
display(pd.Series(quality_report, name='value').to_frame())

## 4. Generate real EDA tables and figures

In [ ]:
# Charts use only metadata and images observed in this runtime.
dataset_summary = create_dataset_report(audited_data, quality_report, OUTPUT_DIR)
display(dataset_summary)

## 5. Class distribution

In [ ]:
display(DisplayImage(filename=str(OUTPUT_DIR / 'class_distribution.png')))

## 6. Representative validated image samples

In [ ]:
sample_figure = OUTPUT_DIR / 'sample_images.png'
if sample_figure.is_file():
    display(DisplayImage(filename=str(sample_figure)))
else:
    print('No sample figure: this runtime has no readable local images.')

## 7. Image-size distribution

In [ ]:
dimension_figure = OUTPUT_DIR / 'image_dimensions.png'
if dimension_figure.is_file():
    display(DisplayImage(filename=str(dimension_figure)))
else:
    print('No dimension figure: this runtime has no readable local images.')

## 8. Missing, corrupt and unsupported image summary

In [ ]:
problem_rows = audited_data[audited_data['validation_status'].ne('valid')]
display(problem_rows[['image_path', 'label', 'original_partition', 'validation_status', 'validation_message']].head(50))
print(json.dumps({key: quality_report[key] for key in ['metadata_records', 'records_with_accessible_images', 'accessible_image_files', 'valid_image_records', 'valid_images', 'missing_files', 'corrupt_files', 'unsupported_files', 'ambiguous_records', 'invalid_paths']}, indent=2))

## 9. Duplicate and patient-ID leakage checks

In [ ]:
leakage_summary = {
    key: quality_report[key]
    for key in [
        'duplicate_metadata_records', 'duplicate_image_path_records',
        'exact_duplicate_groups', 'perceptual_match_groups',
        'patient_overlap_count', 'image_path_label_conflicts',
        'exact_content_label_conflicts',
    ]
}
display(pd.Series(leakage_summary, name='count').to_frame())
print('Warnings:')
for warning in quality_report['warnings']:
    print('-', warning)
print('Phase 3 must resolve flagged leakage risks before splitting or training.')